In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import findspark

findspark.init()

spark = SparkSession.builder \
        .appName("Practise") \
        .getOrCreate()

In [2]:
path = r"C:\Users\areeb\Downloads\fifa data.csv"

df = spark.read.csv(path, header=True, inferSchema=True)

df.show(5)

+---------+--------------------+-----------------+--------------------+---+----------+---------+---------+-----------+-------------------+-------+---------+---------+--------+----------------+--------------+------------------------+---------+-----------+-------------+----------+---------+------------------+--------------------+-------------+------------------+-----------+----------+--------------------+---------------+--------------------+----+--------+-------+---------+---------+------+---------+-----------+----------+-----------+--------+--------------+--------------------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+-----------------------+-------

In [3]:
df.count()

18278

In [4]:
len(df.columns)

104

In [5]:
df.printSchema()

root
 |-- sofifa_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- nationality: string (nullable = true)
 |-- club: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- preferred_foot: string (nullable = true)
 |-- international_reputation: integer (nullable = true)
 |-- weak_foot: integer (nullable = true)
 |-- skill_moves: integer (nullable = true)
 |-- work_rate: string (nullable = true)
 |-- body_type: string (nullable = true)
 |-- real_face: string (nullable = true)
 |-- release_clause_eur: integer (nullable = true)
 |-- player_tags: st

In [6]:
df.describe().show()

+-------+-----------------+--------------------+-----------+--------------------+------------------+------------------+-----------------+-----------+--------------------+-----------------+----------------+-----------------+------------------+----------------+--------------+------------------------+------------------+------------------+-------------+---------+---------+-------------------+-----------+-------------+------------------+--------------------+--------------------+---------------+--------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+--------------------+------------------+-------------------+--------------------------+-----------------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+-------------------

In [7]:
df.summary().show()

+-------+-----------------+--------------------+-----------+--------------------+------------------+------------------+-----------------+-----------+--------------------+-----------------+----------------+-----------------+------------------+----------------+--------------+------------------------+------------------+------------------+-------------+---------+---------+-------------------+-----------+-------------+------------------+--------------------+--------------------+---------------+--------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+--------------------+------------------+-------------------+--------------------------+-----------------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+-------------------

In [8]:
df.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
).show()

+---------+----------+----------+---------+---+---+---------+---------+-----------+----+-------+---------+---------+--------+----------------+--------------+------------------------+---------+-----------+---------+---------+---------+------------------+-----------+-------------+------------------+-----------+------+--------------------+---------------+--------------------+----+--------+-------+---------+---------+------+---------+-----------+----------+-----------+--------+--------------+-------------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+-----------------------+---------------------+----------------+-------------------+-------------------+---

In [39]:
num_col = [ col_name for col_name , col_type in df.dtypes if col_type in ('int', 'double', 'float', 'bigint')]

means = df.select(
    [F.mean(c).alias(c) for c in num_col]
).collect()[0].asDict()

df = df.fillna(means)

In [44]:
df.select(
    [F.sum( when(col(i).isNull() , 1).otherwise(0) ).alias(i) for i in df.columns]
).show()

+---------+----------+----------+---------+---+---+---------+---------+-----------+----+-------+---------+---------+--------+----------------+--------------+------------------------+---------+-----------+---------+---------+---------+------------------+-----------+-------------+------------------+-----------+------+--------------------+---------------+--------------------+----+--------+-------+---------+---------+------+---------+-----------+----------+-----------+--------+--------------+-------------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+-----------------------+---------------------+----------------+-------------------+-------------------+---

In [69]:
cat_col = ['team_position', 'player_traits', 'nation_position']

for i in cat_col:

    mode = df.filter( F.col(i).isNotNull())\
              .groupBy( F.col(i) )\
             .count()\
             .orderBy('count', ascending = False)\
             .limit(1)\
             .collect()[0][0]
    print(f'Filling column : {i} with mode: {mode}')
    df = df.fillna( value = mode , subset= [i])

Filling column : team_position with mode: SUB
Filling column : player_traits with mode: Early Crosser
Filling column : nation_position with mode: SUB


In [72]:
df.select(
    [F.sum( when(F.col(i).isNull(), 1).otherwise(0) ).alias(i) for i in df.columns]
).show()

+---------+----------+----------+---------+---+---+---------+---------+-----------+----+-------+---------+---------+--------+----------------+--------------+------------------------+---------+-----------+---------+---------+---------+------------------+-----------+-------------+------------------+-----------+------+--------------------+---------------+--------------------+----+--------+-------+---------+---------+------+---------+-----------+----------+-----------+--------+--------------+-------------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+-----------------------+---------------------+----------------+-------------------+-------------------+---

In [9]:
df.groupBy('nationality').count().orderBy('count', ascending=False).show(10)

+-----------+-----+
|nationality|count|
+-----------+-----+
|    England| 1667|
|    Germany| 1216|
|      Spain| 1035|
|     France|  984|
|  Argentina|  886|
|     Brazil|  824|
|      Italy|  732|
|   Colombia|  591|
|      Japan|  453|
|Netherlands|  416|
+-----------+-----+
only showing top 10 rows


In [10]:
df.groupBy('nationality').agg( 
    F.countDistinct('short_name').alias('Unique_Players') 
    )\
    .orderBy('Unique_Players', ascending = False).show(5)

+-----------+--------------+
|nationality|Unique_Players|
+-----------+--------------+
|    England|          1584|
|    Germany|          1204|
|      Spain|           998|
|     France|           973|
|  Argentina|           851|
+-----------+--------------+
only showing top 5 rows


In [11]:
players_salary = df.select("nationality" ,"short_name", "wage_eur")

players_salary.orderBy('wage_eur', ascending = False).show(5)

+-----------+-----------------+--------+
|nationality|       short_name|wage_eur|
+-----------+-----------------+--------+
|  Argentina|         L. Messi|  565000|
|    Belgium|        E. Hazard|  470000|
|   Portugal|Cristiano Ronaldo|  405000|
|    Belgium|     K. De Bruyne|  370000|
|     France|     A. Griezmann|  370000|
+-----------+-----------------+--------+
only showing top 5 rows


In [12]:
df.groupBy("nationality").agg(
    F.count('short_name') ,
    F.count_distinct('short_name') ,
    F.sum('wage_eur') ,
    F.avg('wage_eur') ,
    F.min('wage_eur') ,
    F.max('wage_eur') ,
    F.stddev('wage_eur') 
).orderBy('sum(wage_eur)', ascending = False).show(5)

+-----------+-----------------+--------------------------+-------------+------------------+-------------+-------------+------------------+
|nationality|count(short_name)|count(DISTINCT short_name)|sum(wage_eur)|     avg(wage_eur)|min(wage_eur)|max(wage_eur)|  stddev(wage_eur)|
+-----------+-----------------+--------------------------+-------------+------------------+-------------+-------------+------------------+
|      Spain|             1035|                       998|     16279000|15728.502415458937|         1000|       300000|30598.838001140844|
|    England|             1667|                      1584|     15691000| 9412.717456508699|         1000|       255000|18847.114254448647|
|     France|              984|                       973|     13802000|14026.422764227642|         1000|       370000| 30113.00035608721|
|     Brazil|              824|                       761|     13717000|16646.844660194176|         1000|       290000| 28787.57477589353|
|    Germany|             1

In [13]:
shooting = df.select( 'short_name', 'shooting')

shooting.groupBy('short_name').agg( 
    F.sum('shooting').alias('total_shooting')
).orderBy('total_shooting', ascending = False).show(5)

+------------+--------------+
|  short_name|total_shooting|
+------------+--------------+
|    Paulinho|           585|
|J. Rodríguez|           488|
|J. Hernández|           411|
|    M. Gómez|           331|
| J. Williams|           323|
+------------+--------------+
only showing top 5 rows


In [14]:
defending = df.select("short_name","defending","nationality","club")

defending.groupBy("short_name","nationality","club").agg( 
    F.sum('defending').alias('total_defending')
).orderBy('total_defending', ascending = False).show(5)

+------------+-----------+------------+---------------+
|  short_name|nationality|        club|total_defending|
+------------+-----------+------------+---------------+
|Y. Takahashi|      Japan|  Sagan Tosu|            124|
|  S. Ohlsson|     Sweden|IFK Göteborg|            110|
|G. Chiellini|      Italy|    Juventus|             90|
| V. van Dijk|Netherlands|   Liverpool|             90|
|K. Koulibaly|    Senegal|      Napoli|             89|
+------------+-----------+------------+---------------+
only showing top 5 rows


In [15]:
FCB = df.filter(
    col('club') == 'FC Barcelona'
)

FCB.groupBy(['club','short_name']).agg(
    F.count('wage_eur') , 
    F.sum('wage_eur') , 
    F.avg('wage_eur') , 
    F.max('wage_eur') , 
    F.min('wage_eur') , 
).orderBy('sum(wage_eur)', ascending = False).show()

+------------+---------------+---------------+-------------+-------------+-------------+-------------+
|        club|     short_name|count(wage_eur)|sum(wage_eur)|avg(wage_eur)|max(wage_eur)|min(wage_eur)|
+------------+---------------+---------------+-------------+-------------+-------------+-------------+
|FC Barcelona|       L. Messi|              1|       565000|     565000.0|       565000|       565000|
|FC Barcelona|   A. Griezmann|              1|       370000|     370000.0|       370000|       370000|
|FC Barcelona|      L. Suárez|              1|       355000|     355000.0|       355000|       355000|
|FC Barcelona|Sergio Busquets|              1|       300000|     300000.0|       300000|       300000|
|FC Barcelona|          Piqué|              1|       285000|     285000.0|       285000|       285000|
|FC Barcelona|  M. ter Stegen|              1|       250000|     250000.0|       250000|       250000|
|FC Barcelona|     I. Rakitić|              1|       245000|     245000.0

In [16]:
FCB = df.filter(
    col('club') == 'Real Madrid'
)

FCB.groupBy(['club','short_name']).agg(
    F.count('wage_eur') , 
    F.sum('wage_eur') , 
    F.avg('wage_eur') , 
    F.max('wage_eur') , 
    F.min('wage_eur') , 
).orderBy('sum(wage_eur)', ascending = False).show()

+-----------+---------------+---------------+-------------+-------------+-------------+-------------+
|       club|     short_name|count(wage_eur)|sum(wage_eur)|avg(wage_eur)|max(wage_eur)|min(wage_eur)|
+-----------+---------------+---------------+-------------+-------------+-------------+-------------+
|Real Madrid|      E. Hazard|              1|       470000|     470000.0|       470000|       470000|
|Real Madrid|      L. Modrić|              1|       340000|     340000.0|       340000|       340000|
|Real Madrid|       T. Kroos|              1|       330000|     330000.0|       330000|       330000|
|Real Madrid|   Sergio Ramos|              1|       300000|     300000.0|       300000|       300000|
|Real Madrid|     K. Benzema|              1|       285000|     285000.0|       285000|       285000|
|Real Madrid|        G. Bale|              1|       250000|     250000.0|       250000|       250000|
|Real Madrid|           Isco|              1|       245000|     245000.0|       24

In [17]:
from pyspark.sql.window import Window

w = Window.partitionBy('club').orderBy( col('wage_eur').desc() )

df.withColumn(
    'rank', F.dense_rank().over( w )
).filter(
    col('rank') == 1
).select(
    'club', 'short_name', 'wage_eur', 'rank'
).orderBy(col('wage_eur').desc()).show()

+--------------------+-----------------+--------+----+
|                club|       short_name|wage_eur|rank|
+--------------------+-----------------+--------+----+
|        FC Barcelona|         L. Messi|  565000|   1|
|         Real Madrid|        E. Hazard|  470000|   1|
|            Juventus|Cristiano Ronaldo|  405000|   1|
|     Manchester City|     K. De Bruyne|  370000|   1|
| Paris Saint-Germain|        Neymar Jr|  290000|   1|
|   Manchester United|         P. Pogba|  250000|   1|
|           Liverpool|         M. Salah|  240000|   1|
|             Chelsea|         N. Kanté|  235000|   1|
|   FC Bayern München|   R. Lewandowski|  235000|   1|
|   Tottenham Hotspur|          H. Kane|  220000|   1|
|             Arsenal|    P. Aubameyang|  205000|   1|
|   Borussia Dortmund|          M. Reus|  170000|   1|
|              Napoli|     K. Koulibaly|  150000|   1|
|               Inter|         D. Godín|  135000|   1|
|     Atlético Madrid|         J. Oblak|  125000|   1|
|     West

In [18]:
from pyspark.sql.window import Window

w = Window.partitionBy('nationality').orderBy( col('wage_eur').desc() )

df.withColumn(
    'rank', F.dense_rank().over( w )
).filter(
    col('rank') == 1
).select(
    'nationality', 'short_name', 'wage_eur', 'rank'
).orderBy(col('wage_eur').desc()).show()

+-----------+-----------------+--------+----+
|nationality|       short_name|wage_eur|rank|
+-----------+-----------------+--------+----+
|  Argentina|         L. Messi|  565000|   1|
|    Belgium|        E. Hazard|  470000|   1|
|   Portugal|Cristiano Ronaldo|  405000|   1|
|     France|     A. Griezmann|  370000|   1|
|    Uruguay|        L. Suárez|  355000|   1|
|    Croatia|        L. Modrić|  340000|   1|
|    Germany|         T. Kroos|  330000|   1|
|      Spain|     Sergio Ramos|  300000|   1|
|      Spain|  Sergio Busquets|  300000|   1|
|     Brazil|        Neymar Jr|  290000|   1|
|    England|      R. Sterling|  255000|   1|
|      Wales|          G. Bale|  250000|   1|
|      Egypt|         M. Salah|  240000|   1|
|     Poland|   R. Lewandowski|  235000|   1|
|   Colombia|     J. Rodríguez|  225000|   1|
|    Senegal|          S. Mané|  220000|   1|
|      Italy|     G. Chiellini|  215000|   1|
|    Denmark|       C. Eriksen|  205000|   1|
|      Chile|         A. Vidal|  2

In [77]:
df.withColumn(
    "Salary_Flag" , when( F.col('wage_eur') >= 500000 , "High")\
                   .when( F.col('wage_eur') >= 300000 , "Medium")\
                   .when( F.col('wage_eur') >= 200000 , "Low")\
                   .otherwise("ver_Low")
).select(
    'wage_eur',
    'Salary_Flag'
).show(10)

+--------+-----------+
|wage_eur|Salary_Flag|
+--------+-----------+
|  565000|       High|
|  405000|     Medium|
|  290000|        Low|
|  125000|    ver_Low|
|  470000|     Medium|
|  370000|     Medium|
|  250000|        Low|
|  200000|        Low|
|  340000|     Medium|
|  240000|        Low|
+--------+-----------+
only showing top 10 rows
